# Stage 1 — PDF Ingestion Pipeline
### Space Flight AI — Dataset Builder

This notebook:
1. Parses all PDFs in a folder using Docling
2. Detects scanned vs digital PDFs automatically
3. Exports clean markdown to `parsed_docs.json`
4. Feeds parsed text into an LLM to generate structured decision+reasoning training pairs
5. Saves final dataset as `training_pairs.jsonl` ready for fine-tuning

---
**Folder structure expected:**
```
project/
├── pdfs/                  ← drop all your downloaded PDFs here
│   ├── nasa_apollo11.pdf
│   ├── orbital_mechanics.pdf
│   └── ...
├── parsed_docs.json       ← auto generated after Step 2
└── training_pairs.jsonl   ← auto generated after Step 4
```

## Step 0 — Install Dependencies

In [1]:
# Run once — installs all required packages
!pip install docling pymupdf openai tqdm --quiet

# If running on university H100 cluster, you may need:
# !pip install docling pymupdf openai tqdm --quiet --break-system-packages


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


## Step 1 — Imports and Config

In [ ]:
import os
import json
import fitz                          # PyMuPDF — only used for OCR detection
from pathlib import Path
from tqdm import tqdm
from docling.document_converter import DocumentConverter
from docling.datamodel.pipeline_options import PdfPipelineOptions
from openai import OpenAI            # using OpenAI-compatible API

# ─────────────────────────────────────────────
# CONFIG — edit these paths before running
# ─────────────────────────────────────────────
PDF_FOLDER       = "./data"                 # folder where you drop your PDFs
PARSED_OUTPUT    = "./parsed_docs.json"     # intermediate parsed markdown
TRAINING_OUTPUT  = "./training_pairs.jsonl" # final fine-tuning dataset

# LLM API config — used to generate training pairs from parsed text
# You can use any OpenAI-compatible endpoint
# Options: OpenAI GPT-4, local Ollama, Together AI, etc.
LLM_API_KEY      = "your_openai_key_here"      # replace with your key
LLM_BASE_URL     = "https://api.openai.com/v1"  # or your local endpoint
LLM_MODEL        = "gpt-4o"                 # model used to generate pairs

# How many training pairs to generate per document chunk
PAIRS_PER_CHUNK  = 5

# Chunk size in characters — how much text fed to LLM at once
# 3000 chars ≈ ~750 tokens — safe for most models
CHUNK_SIZE       = 3000

print("✓ Config loaded")
print(f"  PDF folder     : {PDF_FOLDER}")
print(f"  Parsed output  : {PARSED_OUTPUT}")
print(f"  Training output: {TRAINING_OUTPUT}")

2026-05-30 17:11:41.308546: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-30 17:11:41.428821: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780141301.465921  827994 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780141301.477826  827994 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-30 17:11:41.560750: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

✓ Config loaded
  PDF folder     : /home/navya/Desktop/space/data
  Parsed output  : ./parsed_docs.json
  Training output: ./training_pairs.jsonl


## Step 2 — Parse All PDFs with Docling

Automatically detects whether each PDF is:
- **Digital** (modern NASA/ESA docs) → parsed directly
- **Scanned** (Apollo-era docs) → OCR enabled automatically

In [5]:
# ADD THESE at the top of the cell with other imports
from docling.datamodel.base_models import InputFormat
from docling.document_converter import PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions

def needs_ocr(pdf_path: str) -> bool:
    """
    Detect if a PDF is scanned (image-based) vs digital.
    If the first page has almost no extractable text → it's scanned.
    """
    try:
        doc = fitz.open(pdf_path)
        first_page_text = doc[0].get_text().strip()
        doc.close()
        return len(first_page_text) < 50
    except Exception:
        return True  # if in doubt, use OCR


def get_converter(ocr: bool) -> DocumentConverter:
    if ocr:
        options = PdfPipelineOptions(do_ocr=True)
        return DocumentConverter(
            format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=options)}
        )
    return DocumentConverter()


def parse_pdf(pdf_path: str) -> dict:
    """
    Parse a single PDF.
    Returns dict with filename, ocr_used, and markdown text.
    """
    ocr = needs_ocr(pdf_path)
    converter = get_converter(ocr)
    result = converter.convert(pdf_path)
    markdown = result.document.export_to_markdown()
    return {
        "filename": Path(pdf_path).name,
        "ocr_used": ocr,
        "char_count": len(markdown),
        "text": markdown
    }


def parse_all_pdfs(folder: str) -> dict:
    """
    Parse every PDF in the folder.
    Returns dict keyed by filename stem.
    Skips files that fail gracefully.
    """
    pdf_files = list(Path(folder).glob("*.pdf"))

    if not pdf_files:
        print(f"No PDFs found in {folder}")
        print("    Drop your PDF files there and re-run this cell.")
        return {}

    print(f"Found {len(pdf_files)} PDF(s) — starting parse...\n")
    parsed = {}

    for pdf in tqdm(pdf_files, desc="Parsing PDFs"):
        try:
            result = parse_pdf(str(pdf))
            parsed[pdf.stem] = result
            ocr_tag = "[OCR]" if result["ocr_used"] else "[digital]"
            print(f"  ✓ {pdf.name} {ocr_tag} — {result['char_count']:,} chars")
        except Exception as e:
            print(f"  ✗ {pdf.name} — FAILED: {e}")

    return parsed


# ── RUN ──
os.makedirs(PDF_FOLDER, exist_ok=True)
parsed_docs = parse_all_pdfs(PDF_FOLDER)

if parsed_docs:
    with open(PARSED_OUTPUT, "w") as f:
        json.dump(parsed_docs, f, indent=2)
    print(f"\n✓ Saved parsed docs → {PARSED_OUTPUT}")
    print(f"  Total documents parsed: {len(parsed_docs)}")
    total_chars = sum(d['char_count'] for d in parsed_docs.values())
    print(f"  Total characters     : {total_chars:,}")

Found 17 PDF(s) — starting parse...



Parsing PDFs:   0%|          | 0/17 [00:00<?, ?it/s]

Parsing PDFs:   6%|▌         | 1/17 [00:02<00:41,  2.59s/it]

  ✗ 2410.22595v1.pdf — FAILED: module 'torch' has no attribute 'get_default_device'


Parsing PDFs:  12%|█▏        | 2/17 [00:04<00:35,  2.39s/it]

...

  ✗ Rocket Propulsion Elements.pdf — FAILED: module 'torch' has no attribute 'get_default_device'


Parsing PDFs:  94%|█████████▍| 16/17 [00:41<00:02,  2.40s/it]

  ✗ Introduction to Orbital Mechanics and Spacecraft Attitudes for Thermal Engineers CHARTS PDF.pdf — FAILED: module 'torch' has no attribute 'get_default_device'


Parsing PDFs: 100%|██████████| 17/17 [00:43<00:00,  2.58s/it]

  ✗ 19660016018.pdf — FAILED: module 'torch' has no attribute 'get_default_device'


## Step 3 — Inspect Parsed Output

Quick sanity check before feeding to LLM — make sure the text looks clean.

In [ ]:
# Load parsed docs if not already in memory
with open(PARSED_OUTPUT) as f:
    parsed_docs = json.load(f)

# Show summary table
print(f"{'Document':<40} {'OCR':<8} {'Chars':>10}")
print("-" * 62)
for name, doc in parsed_docs.items():
    ocr_tag = "yes" if doc["ocr_used"] else "no"
    print(f"{name[:40]:<40} {ocr_tag:<8} {doc['char_count']:>10,}")

# Preview first 1000 chars of the first document
print("\n" + "=" * 62)
print("PREVIEW — first document, first 1000 characters:")
print("=" * 62)
first_doc = list(parsed_docs.values())[0]
print(first_doc["text"][:1000])

## Step 4 — Generate Training Pairs with LLM

Each parsed document is split into chunks.
Each chunk is sent to the LLM with a structured prompt.
The LLM returns decision+reasoning pairs in JSON format.

**Output format per pair:**
```json
{
  "mission_phase": "Gravity Turn Ascent",
  "situation": "Vehicle at 15km altitude, velocity 350 m/s, dynamic pressure 28 kPa",
  "decision": "Reduce throttle to 70%, maintain current pitch angle",
  "chain_of_thought": "Dynamic pressure approaching max-Q...",
  "theory_reference": "NASA ascent profile guidelines — Max-Q management",
  "source_document": "nasa_apollo11.pdf"
}
```

In [ ]:
# ── LLM CLIENT SETUP ──
client = OpenAI(
    api_key=LLM_API_KEY,
    base_url=LLM_BASE_URL
)


SYSTEM_PROMPT = """
You are a space flight expert building a training dataset for an AI rocket pilot.

You will receive a chunk of text from a space document — this could be theory, 
equations, procedures, mission reports, design notes, or operational guidelines.

Your job is to extract as many USEFUL training pairs as possible from whatever 
is in the text. Be creative and flexible — not every pair needs to be a direct 
flight decision. The AI pilot needs to understand space deeply.

Generate pairs in ANY of these formats depending on what the text contains:

FORMAT A — Flight Decision (when text has operational content)
  situation     : telemetry or scenario requiring a decision
  decision      : what the pilot should do
  chain_of_thought : physics reasoning behind the decision
  theory_reference : principle involved

FORMAT B — Concept Understanding (when text has theory/principles)
  situation     : "Explain [concept] and when it applies during flight"
  decision      : clear explanation of the concept
  chain_of_thought : deeper physics — why it works this way
  theory_reference : source principle or equation

FORMAT C — What-If Reasoning (when text describes constraints or limits)
  situation     : "What happens if [limit is exceeded or condition changes]?"
  decision      : consequence and correct response
  chain_of_thought : physical reasoning for the consequence
  theory_reference : the governing principle

FORMAT D — Procedure Knowledge (when text has step-by-step content)
  situation     : "What is the correct sequence for [procedure]?"
  decision      : the steps in correct order with rationale
  chain_of_thought : why each step must happen in this order
  theory_reference : the operational principle

Always set mission_phase to the most relevant flight phase this knowledge applies to.
If knowledge applies to multiple phases, pick the most specific one.

Mission phases to choose from:
Pre-Launch, Ignition, Liftoff, Gravity Turn, Max-Q, Stage Separation,
Orbital Insertion, Circularization, Hohmann Transfer, Plane Change,
Rendezvous, Docking, Station Keeping, Deorbit Burn, Re-entry,
Landing Burn, Touchdown, Mission Planning, Abort, General Orbital Mechanics

Output ONLY a valid JSON array. No preamble, no explanation, no markdown fences.
Each element must have exactly these keys:
  mission_phase, situation, decision, chain_of_thought, theory_reference

If the chunk is completely irrelevant to spaceflight return []
"""


def chunk_text(text: str, chunk_size: int = CHUNK_SIZE) -> list:
    """Split text into overlapping chunks to avoid cutting mid-concept."""
    chunks = []
    step = int(chunk_size * 0.85)  # 15% overlap between chunks
    for i in range(0, len(text), step):
        chunk = text[i:i + chunk_size]
        if len(chunk) > 200:  # skip tiny trailing chunks
            chunks.append(chunk)
    return chunks


def generate_pairs_from_chunk(chunk: str, source_name: str) -> list:
    """
    Send one text chunk to the LLM.
    Returns a list of structured training pair dicts.
    """
    user_prompt = f"""
Source document: {source_name}

Text chunk:
---
{chunk}
---

Generate up to {PAIRS_PER_CHUNK} decision+reasoning training pairs from this text.
Return only a JSON array.
"""
    try:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_prompt}
            ],
            temperature=0.3,   # low temp = more factual, less creative
            max_tokens=2000
        )

        raw = response.choices[0].message.content.strip()

        # Strip markdown fences if LLM adds them despite instructions
        raw = raw.replace("```json", "").replace("```", "").strip()

        pairs = json.loads(raw)

        # Inject source document name into each pair
        for p in pairs:
            p["source_document"] = source_name

        return pairs

    except json.JSONDecodeError as e:
        print(f"JSON parse error: {e}")
        return []
    except Exception as e:
        print(f"LLM error: {e}")
        return []


print("✓ LLM pipeline functions defined — ready to run Step 5")

## Step 5 — Run the Full Pipeline

This loops through every parsed document, chunks it, calls the LLM, and saves pairs.

**Cost estimate:** ~$0.002 per chunk with GPT-4o. A 100-page PDF ≈ 50 chunks ≈ $0.10 per document.

In [ ]:
all_pairs = []
failed_chunks = 0

for doc_name, doc_data in tqdm(parsed_docs.items(), desc="Documents"):
    text   = doc_data["text"]
    chunks = chunk_text(text)

    print(f"\n📄 {doc_name} — {len(chunks)} chunks")

    doc_pairs = []
    for i, chunk in enumerate(tqdm(chunks, desc=f"  Chunks", leave=False)):
        pairs = generate_pairs_from_chunk(chunk, doc_name)

        if pairs:
            doc_pairs.extend(pairs)
        else:
            failed_chunks += 1

    all_pairs.extend(doc_pairs)
    print(f"  ✓ Generated {len(doc_pairs)} pairs from this document")

# ── SAVE AS JSONL ──
# JSONL = one JSON object per line — standard format for LLM fine-tuning
with open(TRAINING_OUTPUT, "w") as f:
    for pair in all_pairs:
        f.write(json.dumps(pair) + "\n")

print("\n" + "=" * 50)
print(f"✓ Pipeline complete")
print(f"  Total pairs generated : {len(all_pairs)}")
print(f"  Failed chunks skipped : {failed_chunks}")
print(f"  Saved to              : {TRAINING_OUTPUT}")

## Step 6 — Inspect and Validate Training Pairs

Before fine-tuning, always check what was generated.
Look for: blank fields, hallucinated physics, wrong phase names.

In [ ]:
# Load and display sample pairs
with open(TRAINING_OUTPUT) as f:
    loaded_pairs = [json.loads(line) for line in f]

print(f"Total training pairs loaded: {len(loaded_pairs)}\n")

# Distribution of mission phases
from collections import Counter
phase_counts = Counter(p.get("mission_phase", "unknown") for p in loaded_pairs)
print("Mission phase distribution:")
for phase, count in phase_counts.most_common():
    bar = "█" * (count // max(1, max(phase_counts.values()) // 20))
    print(f"  {phase:<35} {count:>4}  {bar}")

# Show 3 random pairs in full
import random
print("\n" + "=" * 60)
print("SAMPLE PAIRS (3 random):")
for pair in random.sample(loaded_pairs, min(3, len(loaded_pairs))):
    print("\n" + "-" * 60)
    print(f"Phase      : {pair.get('mission_phase')}")
    print(f"Situation  : {pair.get('situation')}")
    print(f"Decision   : {pair.get('decision')}")
    print(f"Reasoning  : {pair.get('chain_of_thought')}")
    print(f"Theory ref : {pair.get('theory_reference')}")
    print(f"Source     : {pair.get('source_document')}")

## Step 7 — Convert to Fine-Tuning Format

Convert raw pairs into the prompt/response format that QLoRA fine-tuning expects.
Uses the special tokens defined for this project.

**Output format:**
```
<|mission_phase|>Gravity Turn<|telemetry|>altitude: 15000m...<|reasoning|>...<|decision|>Reduce throttle to 70%
```

In [ ]:
FINETUNE_OUTPUT = "./finetune_ready.jsonl"

def format_for_finetuning(pair: dict) -> dict:
    """
    Convert a raw training pair into prompt/response format.
    The model learns to read a situation and produce
    structured reasoning + decision output.
    """
    prompt = (
        f"<|mission_phase|>{pair.get('mission_phase', 'Unknown')}\n"
        f"<|situation|>{pair.get('situation', '')}\n"
        f"What is the correct decision and reasoning?"
    )

    response = (
        f"<|reasoning|>{pair.get('chain_of_thought', '')}\n"
        f"<|theory_ref|>{pair.get('theory_reference', '')}\n"
        f"<|decision|>{pair.get('decision', '')}"
    )

    # Standard instruction-tuning format
    return {
        "messages": [
            {"role": "system",    "content": "You are an AI rocket pilot. Analyse the situation and provide your reasoning and decision."},
            {"role": "user",      "content": prompt},
            {"role": "assistant", "content": response}
        ],
        "source": pair.get("source_document", "")
    }


# Convert all pairs
finetune_data = [format_for_finetuning(p) for p in loaded_pairs]

# Save
with open(FINETUNE_OUTPUT, "w") as f:
    for item in finetune_data:
        f.write(json.dumps(item) + "\n")

print(f"✓ Fine-tune ready dataset saved → {FINETUNE_OUTPUT}")
print(f"  Total samples: {len(finetune_data)}")

# Preview one formatted sample
print("\nSAMPLE FORMATTED ENTRY:")
sample = finetune_data[0]
for msg in sample["messages"]:
    print(f"\n[{msg['role'].upper()}]")
    print(msg["content"])

## Step 8 — Dataset Health Check

Final checks before handing off to the fine-tuning notebook.
Flags any pairs with empty fields, too-short reasoning, or missing theory references.

In [ ]:
issues = []
REQUIRED_FIELDS = ["mission_phase", "situation", "decision", "chain_of_thought", "theory_reference"]
MIN_REASONING_CHARS = 100  # chain_of_thought shorter than this is too shallow

for i, pair in enumerate(loaded_pairs):
    for field in REQUIRED_FIELDS:
        if not pair.get(field, "").strip():
            issues.append(f"Pair {i}: missing or empty field '{field}'")

    if len(pair.get("chain_of_thought", "")) < MIN_REASONING_CHARS:
        issues.append(f"Pair {i}: chain_of_thought too short ({len(pair.get('chain_of_thought',''))} chars) — needs more depth")

if issues:
    print(f"⚠️  Found {len(issues)} issues:\n")
    for issue in issues[:20]:  # show first 20
        print(f"  - {issue}")
    if len(issues) > 20:
        print(f"  ... and {len(issues) - 20} more")
    print(f"\n→ These pairs should be reviewed or removed before fine-tuning.")
else:
    print(f"✓ All {len(loaded_pairs)} pairs passed health check")
    print(f"  Dataset is ready to pass to 02_finetune_qwen.ipynb")

# Final stats
avg_reasoning_len = sum(len(p.get("chain_of_thought", "")) for p in loaded_pairs) / max(1, len(loaded_pairs))
print(f"\nDataset stats:")
print(f"  Total pairs          : {len(loaded_pairs)}")
print(f"  Unique sources       : {len(set(p.get('source_document') for p in loaded_pairs))}")
print(f"  Unique phases        : {len(phase_counts)}")
print(f"  Avg reasoning length : {avg_reasoning_len:.0f} chars")

---
## ✓ Pipeline Complete

**Output files:**
| File | Description |
|---|---|
| `parsed_docs.json` | Raw markdown from all PDFs |
| `training_pairs.jsonl` | Structured decision+reasoning pairs |
| `finetune_ready.jsonl` | Formatted for QLoRA fine-tuning |

**Next step:** Open `02_finetune_qwen.ipynb` and load `finetune_ready.jsonl`

**Recommended minimum dataset size before fine-tuning:**
- 500 pairs — basic results
- 2,000 pairs — good generalization  
- 5,000+ pairs — strong domain knowledge